### Week 6, Day 2

We're about to create and use our own MCP Server and MCP Client!

It's pretty simple, but it's not super-simple. The excitment around MCP is about how easy it is to share and use other MCP Servers - making our own does involve a bit of work.

Let's review some python code made mostly by a hard-working Engineering Team:

accounts.py

In [1]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio
from IPython.display import display, Markdown

load_dotenv(override=True)

True

In [2]:
from groceries import Grocery

In [ ]:
grocery = Grocery.get_or_create("eggs")  # Fix 7: use .get(), not Grocery("eggs")    

In [ ]:
grocery.stock(10)

In [ ]:
grocery.consume(4)

In [ ]:
grocery.list_transactions()

In [ ]:
grocery.get_grocery_report()

### Now we write an MCP server and use it directly!

In [3]:
# Now let's use our accounts server as an MCP server

params = {"command": "uv", "args": ["run", "smartfridge_server.py"]}
async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as server:
    mcp_tools = await server.list_tools()


In [ ]:
mcp_tools

In [4]:
instructions = """You are a smart refrigerator that is able to inventory groceries, 
and answer questions about the inventory and the balance available. 

When told about current inventory, you MUST first call stock() for each item, 
then call consume() for any consumed quantities.

I have 10 eggs, 6 bananas and 10 breads. 
I have consumed 2 eggs, 6 bananas and 2 breads."""
request = "What's my balance of eggs, bananas, breads and milk?"
model = "gpt-4.1-mini"

In [5]:

async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="grocery_manager", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("grocery_manager"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1 204 No Content"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1 204 No Content"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"


Your current balance is:
- Eggs: 8 (10 stocked - 2 consumed)
- Bananas: 0 (6 stocked - 6 consumed)
- Breads: 8 (10 stocked - 2 consumed)
- Milk: 0 (no record of stocking or consumption)

Would you like to add milk or any other items to your inventory?

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1 204 No Content"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1 204 No Content"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1 204 No Content"


### Now let's build our own MCP Client

In [6]:
from groceries_client import get_groceries_tools_openai, read_groceries_resource, list_groceries_tools

mcp_tools = await list_groceries_tools()
print(mcp_tools)

[Tool(name='stock', title=None, description='Stock grocery item given the name and quantity.\n\n    Args:\n        name: The name of the grocery\n        quantity: the amount of the grocery\n    ', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}, 'quantity': {'title': 'Quantity', 'type': 'integer'}}, 'required': ['name', 'quantity'], 'title': 'stockArguments', 'type': 'object'}, outputSchema=None, icons=None, annotations=None, meta=None), Tool(name='consume', title=None, description='Consume grocery item given the name and quantity.\n\n    Args:\n        name: The name of the grocery\n        quantity: the amount of the grocery\n    ', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}, 'quantity': {'title': 'Quantity', 'type': 'integer'}}, 'required': ['name', 'quantity'], 'title': 'consumeArguments', 'type': 'object'}, outputSchema=None, icons=None, annotations=None, meta=None), Tool(name='list_transactions', title=None, description='List 

In [7]:
openai_tools = await get_groceries_tools_openai()
print(openai_tools)

[FunctionTool(name='stock', description='Stock grocery item given the name and quantity.\n\n    Args:\n        name: The name of the grocery\n        quantity: the amount of the grocery\n    ', params_json_schema={'properties': {'name': {'title': 'Name', 'type': 'string'}, 'quantity': {'title': 'Quantity', 'type': 'integer'}}, 'required': ['name', 'quantity'], 'title': 'stockArguments', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function get_groceries_tools_openai.<locals>.<lambda> at 0x111a64720>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None), FunctionTool(name='consume', description='Consume grocery item given the name and quantity.\n\n    Args:\n        name: The name of the grocery\n        quantity: the amount of the grocery\n    ', params_json_schema={'properties': {'name': {'title': 'Name', 'type': 'string'}, 'quantity': {'title': 'Quantity', 'type': 'integer'}}, 'required': ['name', 'quantity'], 'title'

In [8]:
request = "What's my balance of eggs, bananas, breads and milk?"

with trace("account_mcp_client"):
    agent = Agent(name="account_manager", instructions=instructions, model=model, tools=openai_tools)
    result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"


Your current balance is:
- Eggs: 8 (10 stocked - 2 consumed)
- Bananas: 0 (6 stocked - 6 consumed)
- Breads: 8 (10 stocked - 2 consumed)
- Milk: 0 (no milk stocked)

Let me know if you need anything else!

In [9]:
context = await read_groceries_resource("eggs")
print(context)

{"name": "eggs", "available_quantity": 14, "added_at": "2026-03-06 14:11:41.349075", "consumed_at": "2026-03-06 14:11:43.441069", "consumed_quantity": 4, "transactions": []}


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercises</h2>
            <span style="color:#ff7800;">Make your own MCP Server! Make a simple function to return the current Date, and expose it as a tool so that an Agent can tell you today's date.<br/>Harder optional exercise: then make an MCP Client, and use a native OpenAI call (without the Agents SDK) to use your tool via your client.
            </span>
        </td>
    </tr>
</table>